# 07 - Análisis de completitud y tratamiento de valores ausentes

Caracteriza los valores ausentes del dataset unificado y define el tratamiento aplicado antes de la fase de modelado.

## Planteamiento

El tratamiento de los valores ausentes debe distinguir entre dos fenómenos de naturaleza distinta:

- **Ausencias puntuales**, de uno a pocos días, atribuibles a incidencias de la instrumentación. Admiten interpolación sin comprometer la validez de la serie.
- **Bloques prolongados**, correspondientes a periodos sin instrumentación o anteriores a la puesta en servicio de la estación. No son imputables y requieren acotar el periodo de análisis o excluir la serie.

Un criterio adicional condiciona el tratamiento de la variable objetivo: la imputación no debe emplear información posterior al instante imputado, dado que las variables derivadas que se construyan sobre ella (retardos, medias móviles) se incorporarán al modelo.

## 1. Configuración

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("..")

DIR_PROCESSED = Path("../data/processed")
DIR_FIGURAS = Path("../outputs/figures")

# Longitud máxima de racha admitida para imputación
MAX_RACHA_IMPUTABLE = 5

dataset = (pd.read_parquet(DIR_PROCESSED / "dataset_unificado.parquet")
           .sort_values(["ID_SAIH", "fecha"])
           .reset_index(drop=True))

experimentales = set(pd.read_parquet(
    DIR_PROCESSED / "embalses_experimento_calidad.parquet")["ID_SAIH"])

print(f"Dataset: {len(dataset):,} filas | {dataset['ID_SAIH'].nunique()} embalses")
print(f"Periodo: {dataset['fecha'].min():%Y-%m-%d} a {dataset['fecha'].max():%Y-%m-%d}")
print(f"Conjunto experimental de calidad: {len(experimentales)} embalses")

Dataset: 230,090 filas | 35 embalses
Periodo: 2006-01-01 a 2023-12-31
Conjunto experimental de calidad: 17 embalses


## 2. Caracterización de las rachas de ausencia

Para cada variable y embalse se identifican las rachas consecutivas de valores ausentes. La distribución de sus longitudes determina qué proporción del problema corresponde a incidencias puntuales y cuál a periodos sin registro.

In [2]:
def rachas_por_variable(df, var):
    """Longitudes de las rachas de ausencia de una variable, por embalse."""
    filas = []
    for emb, g in df.groupby("ID_SAIH"):
        nulo = g[var].isna()
        if not nulo.any():
            continue
        grupo = (nulo != nulo.shift()).cumsum()
        for _, s in g[nulo].groupby(grupo[nulo]):
            filas.append({"ID_SAIH": emb, "dias": len(s),
                          "desde": s["fecha"].min(), "hasta": s["fecha"].max()})
    return pd.DataFrame(filas)

VARIABLES = ["pct_llenado", "aportacion_m3s", "salida_m3s",
             "aemet_precipitacion_mm", "aemet_temp_media_c",
             "aemet_humedad_pct", "aemet_viento_ms"]

resumen = []
rachas = {}
for var in VARIABLES:
    r = rachas_por_variable(dataset, var)
    rachas[var] = r
    n_aus = dataset[var].isna().sum()
    resumen.append({
        "variable": var,
        "ausentes_%": round(dataset[var].isna().mean() * 100, 1),
        "rachas": len(r),
        "cortas_%": round((r["dias"] <= MAX_RACHA_IMPUTABLE).mean() * 100, 0) if len(r) else 0,
        "max_dias": int(r["dias"].max()) if len(r) else 0,
        "dias_en_bloques_largos_%": round(
            r.loc[r["dias"] > MAX_RACHA_IMPUTABLE, "dias"].sum() / n_aus * 100, 0) if n_aus else 0,
    })

print(pd.DataFrame(resumen).to_string(index=False))

              variable  ausentes_%  rachas  cortas_%  max_dias  dias_en_bloques_largos_%
           pct_llenado         3.2    1017      99.0      2191                      85.0
        aportacion_m3s         3.5    1395     100.0      2191                      80.0
            salida_m3s         3.7    1039      97.0      2191                      86.0
aemet_precipitacion_mm         4.0    2220      93.0      1047                      67.0
    aemet_temp_media_c         3.1     992      88.0       817                      79.0
     aemet_humedad_pct         5.9    1367      85.0      1004                      85.0
       aemet_viento_ms        14.2    2204      88.0      6574                      90.0


### Bloques prolongados

Se identifican las rachas superiores al umbral de imputación, que concentran la mayor parte de los días ausentes pese a representar una fracción reducida del número de rachas. Su tratamiento no puede ser la imputación, sino la acotación del periodo o la exclusión de la serie afectada.

In [3]:
largos = pd.concat(
    [r[r["dias"] > MAX_RACHA_IMPUTABLE].assign(variable=v) for v, r in rachas.items()],
    ignore_index=True)

largos["experimental"] = largos["ID_SAIH"].isin(experimentales)

print(f"Bloques prolongados: {len(largos)}\n")
print("=== Los 15 mayores ===")
print(largos.nlargest(15, "dias")[
    ["variable", "ID_SAIH", "dias", "desde", "hasta", "experimental"]]
    .assign(desde=lambda d: d["desde"].dt.date, hasta=lambda d: d["hasta"].dt.date)
    .to_string(index=False))

print("\n=== Embalses del conjunto experimental afectados ===")
afectados = largos[largos["experimental"]].groupby(["ID_SAIH", "variable"])["dias"].sum()
print(afectados.sort_values(ascending=False).head(15).to_string()
      if len(afectados) else "  ninguno")

Bloques prolongados: 813

=== Los 15 mayores ===
              variable ID_SAIH  dias      desde      hasta  experimental
       aemet_viento_ms    E35A  6574 2006-01-01 2023-12-31          True
       aemet_viento_ms    E571  6574 2006-01-01 2023-12-31          True
       aemet_viento_ms    E033  5630 2006-01-01 2021-05-31          True
           pct_llenado    E024  2191 2018-01-01 2023-12-31         False
        aportacion_m3s    E024  2191 2018-01-01 2023-12-31         False
            salida_m3s    E024  2191 2018-01-01 2023-12-31         False
           pct_llenado    E05A  2099 2006-01-01 2011-09-30         False
        aportacion_m3s    E05A  2099 2006-01-01 2011-09-30         False
            salida_m3s    E05A  2099 2006-01-01 2011-09-30         False
        aportacion_m3s    E003  2069 2006-01-01 2011-08-31         False
            salida_m3s    E003  2069 2006-01-01 2011-08-31         False
           pct_llenado    E003  1096 2006-01-01 2008-12-31         False
ae

In [4]:
CONSERVAR = ["pct_llenado", "aportacion_m3s", "salida_m3s",
             "aemet_temp_media_c", "aemet_temp_min_c", "aemet_temp_max_c",
             "aemet_precipitacion_mm", "aemet_humedad_pct"]

sub = dataset[dataset["ID_SAIH"].isin(experimentales)]
print("Cobertura en el conjunto experimental:")
for c in CONSERVAR:
    print(f"  {c:<26} {sub[c].notna().mean()*100:5.1f}%")

print("\nPeor embalse por variable:")
peor = sub.groupby("ID_SAIH")[CONSERVAR].apply(lambda g: g.notna().mean() * 100).min()
print(peor.round(1).to_string())

Cobertura en el conjunto experimental:
  pct_llenado                 99.7%
  aportacion_m3s              99.6%
  salida_m3s                  99.7%
  aemet_temp_media_c          97.7%
  aemet_temp_min_c            97.7%
  aemet_temp_max_c            97.7%
  aemet_precipitacion_mm      96.5%
  aemet_humedad_pct           95.1%

Peor embalse por variable:
pct_llenado               98.6
aportacion_m3s            98.2
salida_m3s                98.4
aemet_temp_media_c        89.2
aemet_temp_min_c          89.2
aemet_temp_max_c          89.2
aemet_precipitacion_mm    85.1
aemet_humedad_pct         76.7


In [5]:
h = sub.groupby("ID_SAIH")["aemet_humedad_pct"].apply(lambda s: s.notna().mean() * 100)
print(h.sort_values().head(5).round(1).to_string())

ID_SAIH
E571    76.7
E35A    81.4
E033    84.3
E028    85.8
E570    96.5


### Variables descartadas por cobertura insuficiente

El análisis de bloques prolongados revela que tres variables meteorológicas presentan series completamente ausentes en algunos embalses del conjunto experimental:

- **Velocidad del viento**: sin registro alguno en E35A y E571 durante los dieciocho años del periodo, y sin datos en E033 hasta 2021.
- **Insolación** y **presión atmosférica**: con coberturas globales del 57,6% y el 64,6% respectivamente.

Las tres se descartan del modelado. La ausencia no responde a incidencias puntuales sino a que las estaciones asignadas a esos embalses no registran dichos parámetros, por lo que no cabe imputación. Se conservan las temperaturas, la precipitación y la humedad relativa, que son además las variables requeridas para el cálculo de la evapotranspiración de referencia abordado en la fase de ingeniería de variables.

In [6]:
DESCARTADAS = ["aemet_viento_ms", "aemet_insolacion_h",
               "aemet_presion_max_hpa", "aemet_presion_min_hpa"]

dataset = dataset.drop(columns=[c for c in DESCARTADAS if c in dataset.columns])
print(f"Variables tras el descarte: {dataset.shape[1]}")
print(f"Columnas: {[c for c in dataset.columns if c.startswith('aemet_')]}")

Variables tras el descarte: 26
Columnas: ['aemet_temp_media_c', 'aemet_temp_min_c', 'aemet_temp_max_c', 'aemet_precipitacion_mm', 'aemet_humedad_pct']


## 3. Tratamiento de los valores ausentes

Se aplican criterios diferenciados según el papel de cada variable en el modelado.

**Variable objetivo.** El porcentaje de llenado se completa mediante arrastre del último valor observado, limitado a rachas de hasta cinco días. Se descarta la interpolación lineal porque emplea el valor posterior al hueco, información que no estaría disponible en el instante de la predicción y que se propagaría a los retardos y medias móviles construidos sobre esta variable. El arrastre es además coherente con la dinámica del llenado, cuya variación diaria es reducida.

**Variables predictoras.** Se interpolan linealmente con el mismo límite de cinco días. En este caso el uso del valor posterior es admisible, ya que en un escenario operativo los registros de las estaciones estarían consolidados en el momento de generar la predicción.

**Bloques prolongados.** Las rachas superiores al límite no se imputan y se mantienen como ausentes, para que su tratamiento sea explícito en la fase de modelado y no quede enmascarado por valores sintéticos.

La imputación se aplica de forma independiente sobre los conjuntos de entrenamiento y de test, separados por el corte del 1 de enero de 2022. De este modo se evita que la interpolación de un valor ausente próximo a la frontera emplee registros pertenecientes al conjunto reservado para la evaluación.

In [11]:
OBJETIVO = "pct_llenado"
PREDICTORAS = [c for c in dataset.columns
               if c not in ("fecha", "ID_SAIH", "Nombre_SAIH", "Sistema",
                            "Capacidad_hm3", OBJETIVO)]

CORTE_TEST = pd.Timestamp("2022-01-01")

def imputar_bloque(df):
    """Imputa un bloque temporal de forma independiente, de modo que ningún valor
    del conjunto de test intervenga en el relleno del conjunto de entrenamiento."""
    d = df.sort_values(["ID_SAIH", "fecha"]).copy()
    d[OBJETIVO] = d.groupby("ID_SAIH")[OBJETIVO].transform(
        lambda s: s.ffill(limit=MAX_RACHA_IMPUTABLE))
    for c in PREDICTORAS:
        d[c] = d.groupby("ID_SAIH")[c].transform(
            lambda s: s.interpolate(method="linear", limit=MAX_RACHA_IMPUTABLE,
                                    limit_area="inside"))
    return d

antes = dataset[[OBJETIVO] + PREDICTORAS].isna().sum()

dataset_imp = (pd.concat([
    imputar_bloque(dataset[dataset["fecha"] < CORTE_TEST]),
    imputar_bloque(dataset[dataset["fecha"] >= CORTE_TEST]),
], ignore_index=True)
    .sort_values(["ID_SAIH", "fecha"])
    .reset_index(drop=True))

despues = dataset_imp[[OBJETIVO] + PREDICTORAS].isna().sum()
comp = pd.DataFrame({"antes": antes, "despues": despues})
comp["imputados"] = comp["antes"] - comp["despues"]
comp["cobertura_%"] = ((1 - comp["despues"] / len(dataset_imp)) * 100).round(1)
print(comp.to_string())

                                antes  despues  imputados  cobertura_%
pct_llenado                      7477     6327       1150         97.3
volumen_hm3                      7477     6332       1145         97.2
aportacion_m3s                   7973     6373       1600         97.2
salida_m3s                       8616     7284       1332         96.8
aemet_temp_media_c               7180     5105       2075         97.8
aemet_temp_min_c                 7173     5105       2068         97.8
aemet_temp_max_c                 7171     5105       2066         97.8
aemet_precipitacion_mm           9196     5404       3792         97.7
aemet_humedad_pct               13472    10407       3065         95.5
cal_arriba_amonio_mgl          181442   177138       4304         23.0
cal_arriba_conductividad_uscm  175194   172051       3143         25.2
cal_arriba_oxigeno_mgl         175682   172218       3464         25.2
cal_arriba_ph                  175088   171806       3282         25.3
cal_ar

In [12]:
sub_imp = dataset_imp[dataset_imp["ID_SAIH"].isin(experimentales)]

print("Cobertura en el conjunto experimental tras imputación:")
for c in [OBJETIVO] + [x for x in PREDICTORAS if not x.startswith("cal_")]:
    print(f"  {c:<26} {sub_imp[c].notna().mean()*100:5.1f}%")

arriba = [c for c in sub_imp.columns if c.startswith("cal_arriba_")]
abajo = [c for c in sub_imp.columns if c.startswith("cal_abajo_")]
print(f"\n  algún dato de calidad        "
      f"{sub_imp[arriba + abajo].notna().any(axis=1).mean()*100:5.1f}%")

Cobertura en el conjunto experimental tras imputación:
  pct_llenado                100.0%
  volumen_hm3                100.0%
  aportacion_m3s             100.0%
  salida_m3s                 100.0%
  aemet_temp_media_c          98.3%
  aemet_temp_min_c            98.3%
  aemet_temp_max_c            98.3%
  aemet_precipitacion_mm      98.2%
  aemet_humedad_pct           95.7%

  algún dato de calidad         95.6%


## 4. Dataset tratado

Se guarda el dataset con los valores ausentes tratados, que constituye la entrada de la fase de ingeniería de variables.

In [13]:
dataset_imp.to_parquet(DIR_PROCESSED / "dataset_imputado.parquet", index=False)

print(f"Guardado: {len(dataset_imp):,} filas x {dataset_imp.shape[1]} columnas")
print(f"Periodo: {dataset_imp['fecha'].min():%Y-%m-%d} a {dataset_imp['fecha'].max():%Y-%m-%d}")

Guardado: 230,090 filas x 26 columnas
Periodo: 2006-01-01 a 2023-12-31
